# Sprint 3 - Tool Calling and MCP

This notebook introduces a small tool layer, structured tool execution errors, a multi-step tool-calling agent, and the MCP server pattern used by the helper core.


## 1. Install the helper core from GitHub

This keeps Colab aligned with the shared repository instead of local notebook-only helpers.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/richhiey/ai-app-dev_Mod-A.git"
WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORK_DIR / "ai-app-dev_Mod-A"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
print(f"Installed helper core from {REPO_URL}")


## 2. Add your OpenRouter key and imports

The manual tool execution cells do not spend model credits. The agent cell does call an enabled OpenRouter chat model.


In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")


In [ ]:
import json

from agent import ToolCallingAgent
from mcp_server import build_mcp_server, course_core_health, keyword_search_documents
from models import ChatModel
from openrouter import OpenRouterClient
from tools import ToolRegistry

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-3")


## 3. Register application tools

A tool is normal application code plus a JSON schema that tells the model how to call it.


In [ ]:
LESSON_NOTES = [
    "Structured output returns validated JSON for routing, UI state, and tool inputs.",
    "Hybrid retrieval combines ChromaDB semantic search with BM25 keyword matching.",
    "Reranking promotes the most useful retrieved passages before generation.",
    "HyDE rewrites a user query into a hypothetical answer document for retrieval.",
    "MCP exposes tools and resources through a common protocol for model clients.",
]


def lesson_lookup(query: str, top_k: int = 2):
    return keyword_search_documents(query=query, documents=LESSON_NOTES, top_k=top_k)


def lab_status(mode: str = "ok"):
    if mode == "timeout":
        raise TimeoutError("The lesson status service did not respond in time.")
    if mode == "missing":
        return {"ok": False, "message": "No lab status is available for that sprint."}
    return {"ok": True, "message": "The lab environment is ready."}


registry = ToolRegistry()
registry.register(
    name="lesson_lookup",
    description="Search short Module A lesson notes for relevant concepts.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query."},
            "top_k": {"type": "integer", "minimum": 1, "maximum": 5},
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    handler=lesson_lookup,
)
registry.register(
    name="lab_status",
    description="Check whether a sprint lab environment is ready.",
    parameters={
        "type": "object",
        "properties": {
            "mode": {"type": "string", "enum": ["ok", "missing", "timeout"]},
        },
        "required": [],
        "additionalProperties": False,
    },
    handler=lab_status,
)


## 4. Inspect the schemas sent to the model

These are OpenRouter-compatible function tool definitions.


In [ ]:
print(json.dumps(registry.to_openrouter_tools(), indent=2))


## 5. Execute a valid tool call

The registry parses JSON arguments, validates required fields, runs the handler, and returns a tool message.


In [ ]:
good_call = {
    "id": "call_lookup_1",
    "type": "function",
    "function": {
        "name": "lesson_lookup",
        "arguments": json.dumps({"query": "hybrid search reranking", "top_k": 2}),
    },
}

result = registry.execute_tool_call(good_call)
print("ok:", result.ok)
print(result.content)
print(result.as_message())


## 6. Handle malformed and failed tool calls

Tool failures are data. The model can read these messages and decide whether to retry, ask for clarification, or continue with a fallback.


In [ ]:
problem_calls = [
    {
        "id": "call_bad_json",
        "type": "function",
        "function": {"name": "lesson_lookup", "arguments": "{\"query\": "},
    },
    {
        "id": "call_extra_arg",
        "type": "function",
        "function": {
            "name": "lesson_lookup",
            "arguments": json.dumps({"query": "MCP", "debug": True}),
        },
    },
    {
        "id": "call_timeout",
        "type": "function",
        "function": {"name": "lab_status", "arguments": json.dumps({"mode": "timeout"})},
    },
]

for call in problem_calls:
    outcome = registry.execute_tool_call(call)
    print(call["id"], "ok=", outcome.ok, "retryable=", outcome.retryable, "error=", outcome.error_type)
    print(outcome.content, "\n")


## 7. Let the model run a multi-step tool loop

The agent sends tool schemas to the model, executes requested calls, appends tool messages, and asks the model to finish.


In [ ]:
agent = ToolCallingAgent(
    client=client,
    tools=registry,
    model=ChatModel.GEMINI_31_FLASH_LITE,
    max_steps=4,
    system_prompt=(
        "You help students with Module A. Use lesson_lookup before answering questions "
        "about specific course concepts. Explain tool errors plainly if they happen."
    ),
)

run = agent.run("Which lesson note should I read to understand why reranking helps RAG?")
print(run.final_content)
print("\nTool calls executed:", len(run.tool_results))
for tool_result in run.tool_results:
    print(tool_result.name, tool_result.ok, tool_result.content[:200])


## 8. Introduce the MCP server pattern

MCP wraps tools behind a standard server interface. The helper core exposes a tiny server so students can inspect the pattern before adding project-specific tools.


In [ ]:
print(course_core_health())

mcp_search = keyword_search_documents(
    query="model clients tools protocol",
    documents=LESSON_NOTES,
    top_k=2,
)
print(mcp_search)

server = build_mcp_server("module-a-demo")
print(type(server).__name__)


## 9. Run the MCP server locally

Colab is not a great place for the interactive MCP Inspector. Run this in a terminal when you want to inspect the server and call its tools:

```bash
git clone https://github.com/richhiey/ai-app-dev_Mod-A.git
cd ai-app-dev_Mod-A
python -m pip install -e ".[dev]"
mcp dev src/mcp_server.py
```

The server exposes `health` and `keyword_search`. In project work, students can add tools next to these and test the schema before wiring them into an agent.


## Checkpoint

Students should be able to explain the difference between direct function calls, model-requested tool calls, tool error messages, and MCP-hosted tools.
